# Classical Forecasting Models

## 1. Import Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

## 2. Load Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"

df_raw = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(df_raw)
df_daily.head()

## 3. Train-Test Split

In [ ]:
target = df_daily["Close"].dropna()
split_idx = int(len(target) * 0.8)

train = target.iloc[:split_idx]
test = target.iloc[split_idx:]

train.shape, test.shape

## 4. Naive Forecast

In [ ]:
naive_forecast = pd.Series(train.iloc[-1], index=test.index, name="Naive")

fig, ax = plt.subplots(figsize=(12, 5))
train.tail(180).plot(ax=ax, label="Train")
test.plot(ax=ax, label="Test")
naive_forecast.plot(ax=ax, label="Naive Forecast")
ax.set_title("Bitcoin Close Price: Naive Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 5. Moving Average Forecast

In [ ]:
moving_average_forecast = (
    target.shift(1)
    .rolling(window=7)
    .mean()
    .reindex(test.index)
    .rename("7-Day Moving Average")
)

fig, ax = plt.subplots(figsize=(12, 5))
train.tail(180).plot(ax=ax, label="Train")
test.plot(ax=ax, label="Test")
moving_average_forecast.plot(ax=ax, label="7-Day Moving Average Forecast")
ax.set_title("Bitcoin Close Price: 7-Day Moving Average Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Evaluation Metrics

In [ ]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
    }


metrics_table = pd.DataFrame(
    [
        evaluate_forecast(test, naive_forecast),
        evaluate_forecast(test, moving_average_forecast),
    ],
    index=["Naive", "7-Day Moving Average"],
)

metrics_table

## 7. Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
test.plot(ax=ax, label="Actual")
naive_forecast.plot(ax=ax, label="Naive Forecast")
moving_average_forecast.plot(ax=ax, label="7-Day Moving Average Forecast")
ax.set_title("Classical Forecast Comparison")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

metrics_table.sort_values("RMSE")

## 8. Key Findings
- The naive forecast provides a simple persistence baseline.
- The 7-day moving average forecast smooths recent price behavior and can be compared against the naive baseline using the metrics table.
- Lower MAE, RMSE, MAPE, and sMAPE values indicate stronger out-of-sample performance on the chronological test set.